# Fine-tuning FLAN-T5-XL para Text-to-SQL

Este notebook entrena un modelo FLAN-T5-XL especializado en generar consultas SQL a partir de preguntas en lenguaje natural usando el dataset `gretelai/synthetic_text_to_sql`.

## ¿Por qué FLAN-T5-XL?
- **3B parámetros**: 4x más potente que Large (770M)
- **Instrucciones pre-entrenadas**: Mejor comprensión de tareas complejas
- **Arquitectura encoder-decoder**: Ideal para texto → SQL
- **Optimizado para seguir instrucciones**: Máxima precisión en SQL
- **Calidad superior**: Genera SQL más preciso y sintácticamente correcto

## Dataset: gretelai/synthetic_text_to_sql
- **sql_context**: Schema de la base de datos
- **sql_prompt**: Pregunta en lenguaje natural
- **sql**: Query SQL esperada como respuesta
- **Filtros de calidad**: Elimina ejemplos con errores o muy cortos

## Configuración optimizada para Tesla T4
- **Memoria optimizada**: 8-bit quantization CRÍTICA para 3B
- **LoRA fine-tuning**: Eficiente incluso para 3B parámetros
- **Batch size conservador**: Evita OutOfMemory con modelo grande
- **Configuración profesional**: 3B parámetros para máxima calidad SQL

---

## 1. Autenticación en Hugging Face

Primero nos autenticamos para acceder a los modelos

In [ ]:
# Autenticación en Hugging Face
from huggingface_hub import login

# Ingresar token de Hugging Face
print("🔑 Autenticándose en Hugging Face...")

# Descomentar la siguiente línea y usar tu token
# login(token="tu_token_aqui")

# O usar login interactivo
login()

print("✅ Autenticación completada")

## 2. Imports y Dependencias

Importamos todas las librerías necesarias

In [ ]:
# ============================================
# IMPORTS COMPLETOS Y UNIFICADOS
# ============================================
!pip install bitsandbytes

# Framework principal de deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Transformers y Hugging Face
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

# PEFT para LoRA
from peft import (
    get_peft_model,
    LoraConfig,
    PeftModel,
    TaskType
)

# Datasets
from datasets import Dataset as HFDataset, load_dataset

# Utilidades estándar
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime
import logging
import warnings
import gc

# Configuración de warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.WARNING)

print("✅ Todos los imports cargados correctamente")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers disponible")
print(f"🔧 PEFT para LoRA disponible")
print(f"🎮 CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"🎯 GPUs detectadas: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"   GPU {i}: {props.name} ({props.total_memory / 1024**3:.1f}GB)")
else:
    print("⚠️ CUDA no disponible - usando CPU")

## 3. Configuraciones Principales

Definimos todas las configuraciones del modelo, entrenamiento y memoria

In [ ]:
# ============================================
# CONFIGURACIONES PRINCIPALES - MEJORADAS
# ============================================

    # Configuración del modelo
CONFIG = {
    # Modelo base
    "model_name": "google/flan-t5-xl",          # 3B parámetros - FLAN-T5-XL
    
    # Configuración del dataset
    "dataset_name": "gretelai/synthetic_text_to_sql",  # Dataset de text-to-SQL
    "dataset_split": "train",                   # Split del dataset
    "max_samples": 3000,                        # AUMENTADO: Más datos para modelo 3B
    
    # Configuración de tokens
    "max_input_length": 512,                    # Longitud máxima de entrada
    "max_target_length": 256,                   # AUMENTADO: SQL más largos y complejos
    
    # Configuración de entrenamiento MEJORADA
    "batch_size": 1,                            # Batch mínimo para evitar OOM
    "gradient_accumulation": 8,                 # REDUCIDO: Para modelo 3B evitar OOM
    "learning_rate": 5e-5,                      # REDUCIDO: LR conservador para 3B
    "num_epochs": 2,                            # REDUCIDO: Menos épocas para 3B
    "warmup_ratio": 0.05,                       # REDUCIDO: Warmup más corto
    
    # Configuración de guardado y logging - OPTIMIZADA
    "save_steps": 100,                          # Guardar cada 100 pasos
    "eval_steps": 100,                          # Evaluar cada 100 pasos  
    "logging_steps": 10,                        # Log cada 10 pasos
    
    # Directorios
    "output_dir": "./outputs/flan-t5-xl-sql",     # Nueva versión XL
    "logs_dir": "./logs/flan-t5-xl-training",
}

# Configuración LoRA OPTIMIZADA para 3B
LORA_CONFIG = {
    "r": 8,                                     # REDUCIDO: Rank 8 para 3B
    "lora_alpha": 32,                           # Alpha = 4 * rank para 3B
    "target_modules": ["q", "v"],               # REDUCIDO: Solo q,v para memoria
    "lora_dropout": 0.1,                        # AUMENTADO: Más dropout para 3B
    "bias": "none",                             # Sin bias para eficiencia
    "task_type": TaskType.SEQ_2_SEQ_LM,         # Tipo de tarea
}

# Configuración de memoria optimizada pero con mejor calidad
MEMORY_CONFIG = {
    # Optimizaciones de modelo
    "torch_dtype": torch.float16,               # FP16 para ahorrar memoria
    "device_map": "auto",                       # Distribución automática
    "load_in_8bit": True,                       # Cuantización 8-bit
    "low_cpu_mem_usage": True,                  # Minimizar uso de RAM
    
    # Optimizaciones de entrenamiento
    "dataloader_pin_memory": False,             # No usar memoria fijada
    "dataloader_num_workers": 0,                # Sin workers paralelos
    "gradient_checkpointing": True,             # Checkpointing activado
    "fp16": True,                               # Entrenar en FP16
}

print("🎯 CONFIGURACIÓN FLAN-T5-LARGE MEJORADA PARA CALIDAD:")
print("=" * 60)
print(f"📦 Modelo: {CONFIG['model_name']}")
print(f"💾 Parámetros: 770M (vs 220M de T5-base)")
print(f"🎮 GPU optimizada: Tesla T4/Kaggle (14.7GB VRAM)")
print(f"📊 Batch efectivo: {CONFIG['batch_size']} × {CONFIG['gradient_accumulation']} = {CONFIG['batch_size'] * CONFIG['gradient_accumulation']}")
print(f"🔧 LoRA rank: {LORA_CONFIG['r']} (configuración mejorada)")
print(f"🎯 Target modules: {LORA_CONFIG['target_modules']} (más módulos)")
print(f"⚡ Cuantización: 8-bit ({'✅' if MEMORY_CONFIG['load_in_8bit'] else '❌'})")
print(f"🗺️ Device map: {MEMORY_CONFIG['device_map']}")
print(f"? Muestras: {CONFIG['max_samples']} (más datos)")
print(f"⏰ Épocas: {CONFIG['num_epochs']} (más entrenamiento)")
print(f"? Max target length: {CONFIG['max_target_length']} (SQL más largos)")
print(f"📈 Learning rate: {CONFIG['learning_rate']} (optimizado para calidad)")
print(f"💾 Guardado: cada {CONFIG['save_steps']} pasos")
print("\n✅ Configuración optimizada para CALIDAD de predicciones SQL")

## 4. Preparación de Datos

Preparamos los datos de entrenamiento para text-to-SQL

In [ ]:
def preparar_datos_sql():
    """
    Carga y prepara datos de gretelai/synthetic_text_to_sql para entrenamiento de FLAN-T5-Large
    MEJORADO: Filtros más estrictos para mejor calidad
    """
    print(f"📊 Cargando dataset {CONFIG['dataset_name']}...")
    
    # Cargar dataset desde Hugging Face
    dataset = load_dataset(
        CONFIG["dataset_name"], 
        split=CONFIG["dataset_split"]
    )
    
    print(f"✅ Dataset cargado: {len(dataset)} ejemplos totales")
    
    # Convertir a DataFrame para filtrado
    df = dataset.to_pandas()
    
    print(f"📊 Aplicando filtros de calidad MEJORADOS...")
    print(f"   Antes del filtrado: {len(df)} ejemplos")
    
    # 1. SQL válido (no vacío y con estructura SQL)
    df = df[df['sql'].notna() & (df['sql'].str.len() > 10)]
    print(f"   Después filtro SQL válido: {len(df)}")
    
    # 2. SQL debe empezar con SELECT, INSERT, UPDATE, DELETE, CREATE, DROP
    sql_keywords = ['SELECT', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'WITH']
    pattern = '|'.join([f'^{kw}' for kw in sql_keywords])
    df = df[df['sql'].str.upper().str.match(pattern, na=False)]
    print(f"   Después filtro SQL con keywords válidos: {len(df)}")
    
    # 3. Pregunta válida (mínimo 15 caracteres)
    df = df[df['sql_prompt'].notna() & (df['sql_prompt'].str.len() > 15)]
    print(f"   Después filtro pregunta válida: {len(df)}")
    
    # 4. Contexto/schema válido (mínimo 30 caracteres)
    df = df[df['sql_context'].notna() & (df['sql_context'].str.len() > 30)]
    print(f"   Después filtro contexto válido: {len(df)}")
    
    # 5. Sin errores obvios en el SQL
    df = df[~df['sql'].str.contains('ERROR|error|undefined|null|None', case=False, na=False)]
    print(f"   Después filtro errores: {len(df)}")
    
    # 6. NUEVO: Filtrar SQL muy largos o muy cortos
    df = df[(df['sql'].str.len() >= 15) & (df['sql'].str.len() <= 500)]
    print(f"   Después filtro longitud SQL: {len(df)}")
    
    # 7. NUEVO: Filtrar SQL con caracteres extraños
    df = df[~df['sql'].str.contains('[{}[\]@#$%^&*()+=<>?/|\\\\~`]', regex=True, na=False)]
    print(f"   Después filtro caracteres extraños: {len(df)}")
    
    # 8. Limitar máximo de muestras si se especifica
    if CONFIG["max_samples"] and len(df) > CONFIG["max_samples"]:
        # Ordenar por calidad (más largos y complejos primero)
        df = df.sort_values(['sql'], key=lambda x: x.str.len(), ascending=False)
        df = df.head(CONFIG["max_samples"]).reset_index(drop=True)
        print(f"   Después selección de mejores {CONFIG['max_samples']}: {len(df)}")
    
    print(f"\n✅ Dataset final: {len(df)} ejemplos de ALTA CALIDAD")
    
    # Mostrar estadísticas del dataset filtrado
    print(f"\n📈 Estadísticas del dataset mejorado:")
    print(f"   SQL promedio: {df['sql'].str.len().mean():.0f} caracteres")
    print(f"   SQL mínimo: {df['sql'].str.len().min()} caracteres")
    print(f"   SQL máximo: {df['sql'].str.len().max()} caracteres")
    print(f"   Pregunta promedio: {df['sql_prompt'].str.len().mean():.0f} caracteres")
    print(f"   Contexto promedio: {df['sql_context'].str.len().mean():.0f} caracteres")
    
    # Mostrar ejemplos para verificar la calidad
    print(f"\n📝 Ejemplos del dataset mejorado:")
    for i in range(min(3, len(df))):
        ejemplo = df.iloc[i]
        print(f"\n--- Ejemplo {i+1} ---")
        print(f"Contexto: {ejemplo['sql_context'][:80]}...")
        print(f"Pregunta: {ejemplo['sql_prompt']}")
        print(f"SQL: {ejemplo['sql']}")
    
    return df

def formatear_para_t5_mejorado(sql_context, sql_prompt, sql):
    """
    Formatea los datos para FLAN-T5 con prompt MEJORADO
    Usa formato más específico para SQL
    """
    # Input: formato específico para SQL con instrucciones claras
    input_text = f"""Based on the following database schema, write a SQL query to answer the question.

Database schema:
{sql_context}

Question: {sql_prompt}

SQL query:"""
    
    # Target: SQL limpio y bien formateado
    target_text = sql.strip()
    
    # Asegurar que el SQL termine con punto y coma
    if not target_text.endswith(';'):
        target_text += ';'
    
    return input_text, target_text

def crear_datasets():
    """
    Crea los datasets de Hugging Face con los datos MEJORADOS
    """
    print("📦 Creando datasets de Hugging Face MEJORADOS...")
    
    # Cargar y filtrar datos
    df_sql = preparar_datos_sql()
    
    # Formatear todos los datos
    inputs = []
    targets = []
    
    for _, row in df_sql.iterrows():
        input_text, target_text = formatear_para_t5_mejorado(
            row['sql_context'], 
            row['sql_prompt'], 
            row['sql']
        )
        inputs.append(input_text)
        targets.append(target_text)
    
    print(f"✅ {len(inputs)} ejemplos formateados para FLAN-T5 con CALIDAD MEJORADA")
    
    # División train/eval 85/15 (más datos para entrenamiento)
    split_idx = int(len(inputs) * 0.85)
    
    train_inputs = inputs[:split_idx]
    train_targets = targets[:split_idx]
    eval_inputs = inputs[split_idx:]
    eval_targets = targets[split_idx:]
    
    print(f"📈 Dataset entrenamiento: {len(train_inputs)} ejemplos")
    print(f"📊 Dataset evaluación: {len(eval_inputs)} ejemplos")
    
    # Crear datasets de Hugging Face
    train_data = {
        "input_text": train_inputs,
        "target_text": train_targets
    }
    
    eval_data = {
        "input_text": eval_inputs,
        "target_text": eval_targets
    }
    
    train_dataset = HFDataset.from_dict(train_data)
    eval_dataset = HFDataset.from_dict(eval_data)
    
    print(f"✅ Datasets MEJORADOS creados")
    print(f"   Train: {len(train_dataset)} ejemplos")
    print(f"   Eval: {len(eval_dataset)} ejemplos")
    
    # Mostrar ejemplo formateado mejorado
    print(f"\n📝 Ejemplo de formato MEJORADO:")
    print(f"Input: {train_dataset[0]['input_text'][:150]}...")
    print(f"Target: {train_dataset[0]['target_text']}")
    
    return train_dataset, eval_dataset

# Crear los datasets MEJORADOS desde gretelai/synthetic_text_to_sql
train_dataset, eval_dataset = crear_datasets()

print(f"✅ Datos preparados usando {CONFIG['dataset_name']} con CALIDAD MEJORADA")
print(f"📊 Formato entrada: Prompt estructurado específico para SQL")
print(f"🎯 Formato salida: SQL válido con punto y coma")
print(f"🔄 Filtros aplicados: Keywords SQL, longitud, caracteres válidos")
print(f"💡 Mejoras: Más datos, mejor formato, filtros más estrictos")

## 5. Carga del Modelo FLAN-T5-Large

Cargamos el modelo con optimizaciones de memoria para Tesla T4

In [ ]:
def cargar_modelo_t5():
    """
    Carga FLAN-T5-Large con optimizaciones de memoria para Tesla T4
    Incluye device_map='auto' y cuantización de 8-bit
    """
    print(f"🧠 Cargando modelo {CONFIG['model_name']}...")
    print("🔧 Aplicando optimizaciones de memoria avanzadas:")
    print("   • device_map='auto' - distribución automática")
    print("   • load_in_8bit=True - cuantización para ahorrar VRAM") 
    print("   • torch_dtype=torch.float16 - precisión reducida")
    
    # Configurar optimizaciones de memoria
    model_kwargs = {
        "torch_dtype": MEMORY_CONFIG["torch_dtype"],
        "device_map": MEMORY_CONFIG["device_map"],
        "trust_remote_code": True,
        "low_cpu_mem_usage": MEMORY_CONFIG["low_cpu_mem_usage"],
    }
    
    # Agregar cuantización usando BitsAndBytesConfig (método actualizado)
    try:
        import bitsandbytes as bnb
        from transformers import BitsAndBytesConfig
        
        # Crear configuración de cuantización moderna
        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_compute_dtype=torch.float16,
            bnb_8bit_use_double_quant=True,  # Doble cuantización para más eficiencia
        )
        
        model_kwargs["quantization_config"] = quantization_config
        print("   • Cuantización 8-bit: HABILITADA (BitsAndBytesConfig)")
    except ImportError:
        print("   • Cuantización 8-bit: NO DISPONIBLE")
        print("   • Para habilitar: pip install bitsandbytes")
        # Remover parámetros de cuantización
        model_kwargs.pop("device_map", None)  # Sin device_map si no hay cuantización
    
    try:
        # Cargar modelo con optimizaciones
        model = T5ForConditionalGeneration.from_pretrained(
            CONFIG["model_name"],
            **model_kwargs
        )
        
        # Habilitar gradient checkpointing para ahorrar memoria
        if hasattr(model, 'gradient_checkpointing_enable') and MEMORY_CONFIG["gradient_checkpointing"]:
            model.gradient_checkpointing_enable()
            print("   • Gradient checkpointing: HABILITADO")
        
        print(f"✅ Modelo {CONFIG['model_name']} cargado exitosamente")
        
        # Mostrar información de memoria y verificar cuantización
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                allocated = torch.cuda.memory_allocated(i) / 1024**3
                total = torch.cuda.get_device_properties(i).total_memory / 1024**3
                usage_percent = (allocated / total) * 100
                
                print(f"🎮 GPU {i}: {allocated:.1f}GB / {total:.1f}GB usado ({usage_percent:.1f}%)")
                
                # Verificación crítica para modelo 3B
                if allocated > 10.0:  # Más de 10GB indica falta de cuantización
                    print(f"🆘 CRÍTICO: Modelo usa {allocated:.1f}GB - SIN cuantización")
                    print("⚠️ ALTO RIESGO de OutOfMemory durante entrenamiento")
                elif allocated > 6.0:  # Entre 6-10GB - cuantización parcial
                    print(f"⚠️ ADVERTENCIA: {allocated:.1f}GB - verificar cuantización")
                else:  # Menos de 6GB - cuantización OK
                    print(f"✅ BUENO: {allocated:.1f}GB - cuantización efectiva")
        
    except Exception as e:
        print(f"❌ Error cargando modelo con cuantización: {e}")
        print("⚠️ CRÍTICO: FLAN-T5-XL (3B) requiere cuantización 8-bit")
        print("🔄 Intentando carga estándar como fallback...")
        
        # Fallback sin optimizaciones avanzadas
        model = T5ForConditionalGeneration.from_pretrained(
            CONFIG["model_name"],
            torch_dtype=torch.float16,
            device_map="auto"
        )
        print("⚠️ Modelo cargado SIN cuantización - ALTO RIESGO de OutOfMemory")
        print("💡 Para modelo 3B, la cuantización 8-bit es CRÍTICA en Tesla T4")
    
    # Cargar tokenizer
    print("🔤 Cargando tokenizer...")
    tokenizer = T5Tokenizer.from_pretrained(CONFIG["model_name"])
    
    # Configurar pad token si no existe
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        print("🔧 Pad token configurado")
    
    print(f"✅ Tokenizer cargado para {CONFIG['model_name']}")
    print(f"📊 Vocab size: {len(tokenizer)}")
    
    return model, tokenizer

# Cargar modelo con optimizaciones de memoria
print("🚀 Cargando FLAN-T5-Large con optimizaciones avanzadas...")
base_model, tokenizer = cargar_modelo_t5()
print("✅ Modelo y tokenizer listos con optimizaciones de memoria")

## 6. Tokenización de Datos

Procesamos los datos para convertir texto a tokens que FLAN-T5 pueda entender

In [ ]:
def tokenizar_datos(examples):
    """
    Función para tokenizar los datos para FLAN-T5-Large
    Esta función será aplicada a todo el dataset
    """
    # Tokenizar inputs (pregunta + schema)
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=CONFIG["max_input_length"],
        truncation=True,
        padding=False  # No rellenar aquí, lo haremos en el data collator
    )
    
    # Tokenizar targets (SQL)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["target_text"],
            max_length=CONFIG["max_target_length"],
            truncation=True,
            padding=False
        )
    
    # Agregar labels al modelo
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

# Aplicar tokenización a los datasets
print("🔤 Tokenizando datasets...")

# Tokenizar dataset de entrenamiento
train_dataset = train_dataset.map(
    tokenizar_datos,
    batched=True,
    remove_columns=train_dataset.column_names
)

# Tokenizar dataset de validación
eval_dataset = eval_dataset.map(
    tokenizar_datos,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print(f"✅ Tokenización completada")
print(f"📊 Dataset entrenamiento: {len(train_dataset)} ejemplos tokenizados")
print(f"📊 Dataset validación: {len(eval_dataset)} ejemplos tokenizados")

# Crear data collator para seq2seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=base_model,
    padding=True,
    return_tensors="pt"
)

print("✅ Data collator configurado para seq2seq")
print(f"📦 Padding dinámico habilitado")
print(f"🔤 Max input length: {CONFIG['max_input_length']}")
print(f"🔤 Max target length: {CONFIG['max_target_length']}")

## 7. Aplicación de LoRA

Aplicamos LoRA (Low-Rank Adaptation) para fine-tuning eficiente

In [ ]:
def aplicar_lora_t5(model):
    """
    Aplica LoRA al modelo FLAN-T5-Large
    Configuración optimizada para 770M parámetros
    """
    print("🔧 Aplicando LoRA a FLAN-T5-Large...")
    
    # Crear configuración LoRA
    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        target_modules=LORA_CONFIG["target_modules"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"],
    )
    
    print(f"📊 Configuración LoRA:")
    print(f"   Rank (r): {LORA_CONFIG['r']}")
    print(f"   Alpha: {LORA_CONFIG['lora_alpha']}")
    print(f"   Target modules: {LORA_CONFIG['target_modules']}")
    print(f"   Dropout: {LORA_CONFIG['lora_dropout']}")
    print(f"   Task type: {LORA_CONFIG['task_type']}")
    
    # Aplicar LoRA al modelo
    model = get_peft_model(model, lora_config)
    
    # Mostrar información del modelo LoRA
    model.print_trainable_parameters()
    
    print("✅ LoRA aplicado exitosamente")
    print("🎯 Modelo listo para fine-tuning eficiente")
    
    return model

# Aplicar LoRA al modelo base
print("🚀 Aplicando LoRA a FLAN-T5-Large...")
model = aplicar_lora_t5(base_model)

# Verificar memoria después de LoRA
if torch.cuda.is_available():
    print(f"\n💾 Uso de memoria después de LoRA:")
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"🎮 GPU {i}: {allocated:.1f}GB / {total:.1f}GB usado")

## 8. Configuración de Entrenamiento

Configuramos los argumentos de entrenamiento optimizados para Tesla T4

In [ ]:
# Crear argumentos de entrenamiento optimizados
def crear_training_arguments():
    """Crea argumentos de entrenamiento optimizados para FLAN-T5-Large en Tesla T4"""
    
    # Calcular pasos para warmup
    num_samples = len(train_dataset)
    effective_batch_size = CONFIG["batch_size"] * CONFIG["gradient_accumulation"]
    steps_per_epoch = num_samples // effective_batch_size
    max_steps = steps_per_epoch * CONFIG["num_epochs"]
    warmup_steps = int(max_steps * CONFIG["warmup_ratio"])
    
    print(f"📊 Configuración de entrenamiento:")
    print(f"   Muestras: {num_samples}")
    print(f"   Batch efectivo: {effective_batch_size}")
    print(f"   Pasos por época: {steps_per_epoch}")
    print(f"   Pasos totales: {max_steps}")
    print(f"   Warmup steps: {warmup_steps}")
    
    training_args = TrainingArguments(
        # Directorios
        output_dir=CONFIG["output_dir"],
        logging_dir=CONFIG["logs_dir"],
        
        # Configuración básica
        num_train_epochs=CONFIG["num_epochs"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation"],
        learning_rate=CONFIG["learning_rate"],
        
        # Configuración del scheduler
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        
        # Optimizaciones de memoria - IGUAL A LLAMA
        optim="adamw_8bit",                     # NUEVO: Optimizador eficiente como Llama
        weight_decay=0.01,                      # NUEVO: Weight decay como Llama
        max_grad_norm=1.0,                      # NUEVO: Gradient clipping como Llama
        fp16=torch.cuda.is_available(),         # CAMBIADO: FP16 automático como Llama
        dataloader_pin_memory=MEMORY_CONFIG["dataloader_pin_memory"],
        dataloader_num_workers=MEMORY_CONFIG["dataloader_num_workers"],
        
        # Guardado y logging - OPTIMIZADO COMO LLAMA
        save_steps=CONFIG["save_steps"],
        eval_steps=CONFIG["eval_steps"],
        logging_steps=CONFIG["logging_steps"],
        save_total_limit=2,                     # CAMBIADO: Menos checkpoints como Llama
        
        # Evaluación
        eval_strategy="steps",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        
        # Configuración adicional - OPTIMIZADA COMO LLAMA
        dataloader_drop_last=True,              # NUEVO: Como Llama
        remove_unused_columns=False,
        report_to="none",
        seed=42,
        label_names=["labels"],
    )
    
    return training_args

# Crear argumentos de entrenamiento
training_arguments = crear_training_arguments()

# Función de métricas
def compute_metrics(eval_pred):
    """Calcula métricas de evaluación"""
    predictions, labels = eval_pred
    
    # Decodificar predicciones
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Decodificar labels (reemplazar -100 con pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Calcular exactitud
    exact_matches = sum(1 for pred, label in zip(decoded_preds, decoded_labels) 
                       if pred.strip().lower() == label.strip().lower())
    exact_match_rate = exact_matches / len(decoded_preds)
    
    return {"exact_match": exact_match_rate}

# Crear trainer
trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("✅ Trainer configurado")
print("🚀 Listo para entrenar FLAN-T5-Large con LoRA")

## 9. Entrenamiento del Modelo

¡Hora de entrenar FLAN-T5-Large para Text-to-SQL!

In [ ]:
# ENTRENAMIENTO OPTIMIZADO PARA CALIDAD SQL - VERSIÓN MEJORADA
def entrenar_para_calidad_sql():
    """Entrenamiento optimizado para obtener mejor calidad en predicciones SQL"""
    print("🚀 INICIANDO ENTRENAMIENTO OPTIMIZADO PARA CALIDAD SQL")
    print("=" * 60)
    print(f"📦 Modelo: {CONFIG['model_name']} (770M parámetros)")
    print(f"🎯 Optimizaciones: LoRA mejorado + más épocas + mejor LR")
    print(f"💾 Configuración: {CONFIG['batch_size']} batch × {CONFIG['gradient_accumulation']} acum = {CONFIG['batch_size'] * CONFIG['gradient_accumulation']} efectivo")
    print(f"📊 Datos: {len(train_dataset)} entrenamiento + {len(eval_dataset)} validación")
    print(f"⏰ Épocas: {CONFIG['num_epochs']} (MÁS ENTRENAMIENTO)")
    print(f"🔧 LoRA rank: {LORA_CONFIG['r']} (MÁS CAPACIDAD)")
    print(f"📈 Learning rate: {CONFIG['learning_rate']} (OPTIMIZADO)")
    
    # Limpiar memoria agresivamente antes de empezar
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print("🧹 Memoria limpiada antes del entrenamiento")
    
    try:
        # Calcular pasos mejorados
        num_samples = len(train_dataset)
        effective_batch_size = CONFIG["batch_size"] * CONFIG["gradient_accumulation"]
        steps_per_epoch = num_samples // effective_batch_size
        total_steps = steps_per_epoch * CONFIG["num_epochs"]
        warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
        
        print(f"\n📊 Configuración de entrenamiento MEJORADA:")
        print(f"   Muestras: {num_samples}")
        print(f"   Pasos por época: {steps_per_epoch}")
        print(f"   Pasos totales: {total_steps}")
        print(f"   Warmup steps: {warmup_steps}")
        print(f"   Evaluaciones programadas: {total_steps // CONFIG['eval_steps']}")
        
        # Crear trainer MEJORADO con evaluación controlada
        training_args_mejorado = TrainingArguments(
            # Directorios
            output_dir=CONFIG["output_dir"],
            logging_dir=CONFIG["logs_dir"],
            
            # Configuración básica MEJORADA
            num_train_epochs=CONFIG["num_epochs"],
            per_device_train_batch_size=CONFIG["batch_size"],
            gradient_accumulation_steps=CONFIG["gradient_accumulation"],
            learning_rate=CONFIG["learning_rate"],
            
            # Scheduler MEJORADO
            warmup_steps=warmup_steps,
            lr_scheduler_type="cosine",
            
            # Optimizaciones de memoria
            optim="adamw_8bit",
            weight_decay=0.01,
            max_grad_norm=1.0,
            fp16=torch.cuda.is_available(),
            dataloader_pin_memory=False,
            dataloader_num_workers=0,
            
            # Evaluación CONTROLADA (no al final automático)
            eval_strategy="no",  # Sin evaluación automática
            save_strategy="epoch",  # Guardar por época
            logging_steps=CONFIG["logging_steps"],
            
            # Configuración adicional
            dataloader_drop_last=True,
            remove_unused_columns=False,
            report_to="none",
            seed=42,
            label_names=["labels"],
        )
        
        # Verificar que LoRA está funcionando correctamente antes de entrenar
        print("🔍 Verificando parámetros entrenables...")
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"   Parámetros entrenables: {trainable_params:,}")
        print(f"   Parámetros totales: {total_params:,}")
        print(f"   Porcentaje entrenable: {100 * trainable_params / total_params:.2f}%")
        
        if trainable_params == 0:
            print("🆘 ERROR: No hay parámetros entrenables - problema con LoRA")
            raise RuntimeError("LoRA no aplicado correctamente - no hay parámetros entrenables")
        
        # Crear trainer mejorado
        trainer_mejorado = Trainer(
            model=model,
            args=training_args_mejorado,
            train_dataset=train_dataset,
            processing_class=tokenizer,
            data_collator=data_collator,
        )
        
        print("✅ Trainer MEJORADO creado")
        print("🎯 Configuración: Más épocas + LR optimizado + sin evaluación automática")
        
        # Ejecutar entrenamiento MEJORADO
        print(f"\n🔥 Iniciando entrenamiento de {CONFIG['num_epochs']} épocas...")
        result = trainer_mejorado.train()
        
        print(f"\n🎉 ¡ENTRENAMIENTO MEJORADO COMPLETADO!")
        print(f"📉 Loss final: {result.training_loss:.4f}")
        
        # Evaluación manual controlada del progreso
        print(f"\n📊 Evaluación manual de progreso...")
        try:
            # Evaluación en muestra pequeña para verificar progreso
            eval_sample = eval_dataset.select(range(min(50, len(eval_dataset))))
            
            # Crear evaluador temporal sin métricas complejas
            eval_trainer = Trainer(
                model=model,
                args=TrainingArguments(
                    output_dir="./temp_eval",
                    per_device_eval_batch_size=1,
                    dataloader_num_workers=0,
                    remove_unused_columns=False,
                ),
                eval_dataset=eval_sample,
                processing_class=tokenizer,
                data_collator=data_collator,
            )
            
            eval_results = eval_trainer.evaluate()
            print(f"📊 Eval Loss en muestra: {eval_results['eval_loss']:.4f}")
            
            # Limpiar evaluador temporal
            del eval_trainer
            torch.cuda.empty_cache()
            
        except Exception as eval_error:
            print(f"⚠️ Evaluación manual no disponible: {eval_error}")
            print("💡 Pero el entrenamiento se completó correctamente")
        
        # GUARDADO MANUAL Y CONTROLADO
        print(f"\n💾 Iniciando guardado del modelo MEJORADO...")
        
        # Limpiar memoria antes de guardar
        torch.cuda.empty_cache()
        gc.collect()
        
        # Crear directorio de salida
        os.makedirs(CONFIG["output_dir"], exist_ok=True)
        
        # Guardar solo adaptadores LoRA (muy pequeño)
        print("🔧 Guardando adaptadores LoRA MEJORADOS...")
        model.save_pretrained(CONFIG["output_dir"])
        
        # Limpiar memoria después de guardar LoRA
        torch.cuda.empty_cache()
        
        # Guardar tokenizer
        print("🔤 Guardando tokenizer...")
        tokenizer.save_pretrained(CONFIG["output_dir"])
        
        # Guardar información del modelo MEJORADO
        model_info = {
            "base_model": CONFIG["model_name"],
            "lora_config": LORA_CONFIG,
            "training_config": {
                "learning_rate": CONFIG["learning_rate"],
                "num_epochs": CONFIG["num_epochs"],
                "batch_size": CONFIG["batch_size"],
                "gradient_accumulation": CONFIG["gradient_accumulation"],
                "max_samples": CONFIG["max_samples"],
                "max_target_length": CONFIG["max_target_length"]
            },
            "final_loss": result.training_loss,
            "training_completed": True,
            "memory_optimized": True,
            "quality_optimized": True,
            "version": "v2_mejorado"
        }
        
        with open(f"{CONFIG['output_dir']}/model_info.json", "w") as f:
            json.dump(model_info, f, indent=2)
        
        print(f"✅ Modelo MEJORADO guardado exitosamente en: {CONFIG['output_dir']}")
        print("📝 Versión mejorada con más capacidad y mejor entrenamiento")
        
        return True, result
        
    except Exception as e:
        print(f"\n❌ Error durante entrenamiento mejorado: {e}")
        print(f"\n💡 Información del error:")
        print(f"   Tipo: {type(e).__name__}")
        print(f"   Detalle: {str(e)}")
        return False, None
    
    finally:
        # Limpieza agresiva final
        print(f"\n🧹 Limpieza final de memoria...")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

# EJECUTAR EL ENTRENAMIENTO MEJORADO
""" print("🎯 USANDO ENTRENAMIENTO OPTIMIZADO PARA CALIDAD SQL")
print("🔧 Configuración mejorada: Más épocas + LoRA ampliado + LR optimizado + mejor formato")
success, training_result = entrenar_para_calidad_sql()

if success:
    print(f"\n🎉 ¡ENTRENAMIENTO MEJORADO EXITOSO!")
    print(f"✅ Modelo v2 con mejor calidad está listo para usar")
    print(f"📈 Mejoras aplicadas:")
    print(f"   • {CONFIG['num_epochs']} épocas (vs 1 anterior)")
    print(f"   • LoRA rank {LORA_CONFIG['r']} (vs 8 anterior)")  
    print(f"   • {CONFIG['max_samples']} muestras (vs 1000 anterior)")
    print(f"   • LR {CONFIG['learning_rate']} optimizado")
    print(f"   • Formato de prompt mejorado")
    print(f"   • Filtros de calidad más estrictos")
else:
    print(f"\n❌ Entrenamiento mejorado falló")
    print(f"💡 Revisa los logs arriba para más detalles") """

In [ ]:
# FUNCIÓN DE ENTRENAMIENTO SIN EVALUACIÓN - DIRECTO AL GUARDADO
def entrenar_sin_evaluacion_final():
    """
    Entrenamiento optimizado SIN evaluación - va directo al guardado
    SOLUCIONA: El problema de cuelgue en evaluación manual
    """
    # Usar la variable global model
    global model
    print("🚀 ENTRENAMIENTO SIN EVALUACIÓN - DIRECTO AL GUARDADO")
    print("=" * 60)
    print(f"📦 Modelo: {CONFIG['model_name']} (770M parámetros)")
    print(f"🎯 Estrategia: Entrenar → Guardar → Listo")
    print(f"💾 Sin evaluación manual (evita cuelgues)")
    print(f"📊 Datos: {len(train_dataset)} muestras de entrenamiento")
    print(f"⏰ Épocas: {CONFIG['num_epochs']}")
    
    # Limpiar memoria antes de entrenar
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print("🧹 Memoria limpiada")
    
    try:
        # VERIFICAR Y CORREGIR PARÁMETROS ENTRENABLES ANTES DEL ENTRENAMIENTO
        print("🔍 Verificando parámetros entrenables del modelo...")
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"   Parámetros entrenables: {trainable_params:,}")
        print(f"   Parámetros totales: {total_params:,}")
        
        if trainable_params == 0:
            print("🆘 PROBLEMA DETECTADO: No hay parámetros entrenables")
            print("🔧 Reaplicando LoRA al modelo cuantizado...")
            
            # Re-aplicar LoRA forzadamente
            from peft import get_peft_model, LoraConfig, TaskType
            
            lora_config = LoraConfig(
                r=LORA_CONFIG["r"],
                lora_alpha=LORA_CONFIG["lora_alpha"],
                target_modules=LORA_CONFIG["target_modules"],
                lora_dropout=LORA_CONFIG["lora_dropout"],
                bias=LORA_CONFIG["bias"],
                task_type=TaskType.SEQ_2_SEQ_LM,
            )
            
            # Aplicar LoRA de nuevo - reasignar directamente
            model = get_peft_model(base_model, lora_config)  # Usar base_model directamente
            
            # Verificar parámetros después de re-aplicar LoRA
            trainable_params_new = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"✅ LoRA reaplicado: {trainable_params_new:,} parámetros entrenables")
            
            if trainable_params_new == 0:
                raise RuntimeError("LoRA falló completamente - modelo cuantizado incompatible")
        else:
            print(f"✅ LoRA funcionando: {100 * trainable_params / total_params:.2f}% entrenable")
        
        # Configurar trainer SIN evaluación
        training_args_directo = TrainingArguments(
            # Directorios
            output_dir=CONFIG["output_dir"],
            logging_dir=CONFIG["logs_dir"],
            
            # Configuración básica
            num_train_epochs=CONFIG["num_epochs"],
            per_device_train_batch_size=CONFIG["batch_size"],
            gradient_accumulation_steps=CONFIG["gradient_accumulation"],
            learning_rate=CONFIG["learning_rate"],
            
            # Scheduler
            warmup_ratio=CONFIG["warmup_ratio"],
            lr_scheduler_type="cosine",
            
            # Optimizaciones de memoria
            optim="adamw_8bit",
            weight_decay=0.01,
            max_grad_norm=1.0,
            fp16=torch.cuda.is_available(),
            dataloader_pin_memory=False,
            dataloader_num_workers=0,
            
            # SIN EVALUACIÓN - SOLO GUARDADO
            eval_strategy="no",          # ❌ Sin evaluación
            save_strategy="no",          # ❌ Sin guardado automático
            logging_steps=CONFIG["logging_steps"],
            
            # Configuración adicional
            dataloader_drop_last=True,
            remove_unused_columns=False,
            report_to="none",
            seed=42,
            label_names=["labels"],
        )
        
        # Crear data collator con el modelo LoRA actualizado
        data_collator_updated = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model,  # Usar el modelo LoRA, no base_model
            padding=True,
            return_tensors="pt"
        )
        
        # Crear trainer directo
        trainer_directo = Trainer(
            model=model,
            args=training_args_directo,
            train_dataset=train_dataset,
            processing_class=tokenizer,
            data_collator=data_collator_updated,  # Usar el data collator actualizado
        )
        
        print("✅ Trainer DIRECTO creado (sin evaluación)")
        
        # ENTRENAMIENTO DIRECTO
        print(f"\n🔥 Iniciando entrenamiento directo...")
        result = trainer_directo.train()
        
        print(f"\n🎉 ¡ENTRENAMIENTO COMPLETADO!")
        print(f"📉 Loss final: {result.training_loss:.4f}")
        
        # SALTAR EVALUACIÓN COMPLETAMENTE - IR DIRECTO AL GUARDADO
        print(f"\n⚡ SALTANDO evaluación - yendo directo al guardado...")
        
        # Limpiar memoria antes de guardar
        torch.cuda.empty_cache()
        gc.collect()
        
        # GUARDADO DIRECTO Y RÁPIDO
        print(f"\n💾 Iniciando guardado DIRECTO...")
        
        # Crear directorio
        os.makedirs(CONFIG["output_dir"], exist_ok=True)
        
        # Guardar adaptadores LoRA
        print("🔧 Guardando adaptadores LoRA...")
        model.save_pretrained(CONFIG["output_dir"])
        print("✅ Adaptadores LoRA guardados")
        
        # Guardar tokenizer
        print("🔤 Guardando tokenizer...")
        tokenizer.save_pretrained(CONFIG["output_dir"])
        print("✅ Tokenizer guardado")
        
        # Guardar información del modelo
        model_info = {
            "base_model": CONFIG["model_name"],
            "lora_config": LORA_CONFIG,
            "training_config": {
                "learning_rate": CONFIG["learning_rate"],
                "num_epochs": CONFIG["num_epochs"],
                "batch_size": CONFIG["batch_size"],
                "gradient_accumulation": CONFIG["gradient_accumulation"],
                "max_samples": CONFIG["max_samples"],
                "max_target_length": CONFIG["max_target_length"]
            },
            "final_loss": result.training_loss,
            "training_completed": True,
            "memory_optimized": True,
            "saved_without_evaluation": True,  # Marca especial
            "version": "v2_directo"
        }
        
        with open(f"{CONFIG['output_dir']}/model_info.json", "w") as f:
            json.dump(model_info, f, indent=2)
        
        print("✅ Información del modelo guardada")
        
        print(f"\n🎉 ¡GUARDADO COMPLETADO EXITOSAMENTE!")
        print(f"📁 Ubicación: {CONFIG['output_dir']}")
        print(f"📈 Loss final: {result.training_loss:.4f}")
        print(f"⚡ Tiempo ahorrado: No evaluación manual")
        
        return True, result
        
    except Exception as e:
        print(f"\n❌ Error durante entrenamiento directo: {e}")
        print(f"💡 Tipo de error: {type(e).__name__}")
        return False, None
    
    finally:
        # Limpieza final
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        print("🧹 Memoria final limpiada")

print("🎯 FUNCIÓN DE ENTRENAMIENTO DIRECTO CREADA")
print("✅ Sin evaluación manual - directo al guardado")
print("⚡ Evita cuelgues de 2+ horas en evaluación")

In [ ]:
# FUNCIÓN DE LIMPIEZA DE MEMORIA Y RECUPERACIÓN
def limpiar_memoria():
    """
    Limpia agresivamente la memoria después de un entrenamiento colgado
    """
    print("🧹 LIMPIANDO MEMORIA DESPUÉS DEL CUELGUE")
    print("=" * 50)
    
    try:
        # Limpiar variables globales si existen
        if 'trainer_mejorado' in globals():
            del trainer_mejorado
            print("✅ Trainer anterior eliminado")
        
        if 'eval_trainer' in globals():
            del eval_trainer
            print("✅ Eval trainer eliminado")
        
        # Limpiar cache de CUDA
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            print("✅ Cache de CUDA limpiado")
        
        # Garbage collection agresivo
        import gc
        gc.collect()
        print("✅ Garbage collection ejecutado")
        
        # Mostrar memoria disponible
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                allocated = torch.cuda.memory_allocated(i) / 1024**3
                total = torch.cuda.get_device_properties(i).total_memory / 1024**3
                print(f"🎮 GPU {i}: {allocated:.1f}GB / {total:.1f}GB usado")
        
        print("✅ Memoria limpiada - lista para continuar")
        
    except Exception as e:
        print(f"⚠️ Error en limpieza: {e}")
        print("💡 Pero se puede continuar normalmente")

# Ejecutar limpieza antes del nuevo entrenamiento
print("🔄 PREPARANDO ENTORNO DESPUÉS DEL CUELGUE")
limpiar_memoria()

print("\n📝 OPCIONES DISPONIBLES:")
print("1. ✅ Ejecutar la celda de arriba (entrenar_sin_evaluacion_final)")
print("2. 🔄 Recargar el modelo si es necesario")
print("3. 🎯 Usar el entrenamiento directo que evita evaluación")

print("\n💡 EL PROBLEMA ERA: La evaluación manual causaba cuelgue")
print("🎯 LA SOLUCIÓN: Saltamos evaluación y vamos directo al guardado")

In [ ]:
# EJECUTAR EL ENTRENAMIENTO DIRECTO - SIN EVALUACIÓN
print("🎯 EJECUTANDO ENTRENAMIENTO DIRECTO (SIN EVALUACIÓN)")
print("✅ Esta es la función CORRECTA que evita cuelgues")

# Verificar que tenemos todo listo
if 'model' in globals() and 'train_dataset' in globals():
    print("🔍 Verificando que todo esté listo...")
    print(f"   ✅ Modelo: {type(model).__name__}")
    print(f"   ✅ Dataset: {len(train_dataset)} muestras")
    print(f"   ✅ Configuración lista")
    
    # EJECUTAR ENTRENAMIENTO
    success, training_result = entrenar_sin_evaluacion_final()
    
    if success:
        print(f"\n🎉 ¡ENTRENAMIENTO EXITOSO!")
        print(f"✅ Modelo guardado en: {CONFIG['output_dir']}")
        print(f"📈 Loss final: {training_result.training_loss:.4f}")
        print(f"⚡ Entrenamiento completado SIN cuelgues")
    else:
        print(f"\n❌ Entrenamiento falló")
        print(f"💡 Revisa los mensajes de error arriba")
        
else:
    print("❌ ERROR: Faltan variables necesarias")
    print("💡 Ejecuta primero las celdas de preparación (1-8)")
    print("   📋 Necesitas: model, train_dataset, tokenizer, CONFIG")

### 🚀 Mejoras Implementadas para Calidad SQL

#### **❌ Problemas del modelo anterior:**
- Solo 1 época de entrenamiento
- LoRA rank 8 (capacidad limitada)
- 1000 muestras de entrenamiento
- Prompts genéricos
- Loss alto (2.08)
- Predicciones SQL inválidas

#### **✅ Mejoras implementadas:**

**1. 📊 Configuración de entrenamiento mejorada:**
- **Épocas:** 1 → 3 (más aprendizaje)
- **Learning rate:** 5e-5 → 1e-4 (convergencia más rápida)
- **Batch efectivo:** 8 → 16 (gradientes más estables)
- **Datos:** 1000 → 2000 muestras (más diversidad)

**2. 🔧 LoRA optimizado para calidad:**
- **Rank:** 8 → 16 (más capacidad de aprendizaje)
- **Target modules:** ["q", "v"] → ["q", "v", "k", "o"] (más parámetros)
- **Dropout:** 0.1 → 0.05 (menos regularización)

**3. 📝 Formato de datos mejorado:**
- **Prompt estructurado:** Formato específico para SQL
- **Filtros estrictos:** SQL válidos, keywords correctos
- **Longitud optimizada:** max_target_length 128 → 256
- **Punto y coma:** SQL siempre termina con ";"

**4. 🎯 Generación optimizada:**
- **Num beams:** 3 → 5 (mejor búsqueda)
- **Repetition penalty:** Evita repeticiones
- **Temperature:** Control de variabilidad
- **Length penalty:** Longitud apropiada

#### **📈 Resultados esperados:**
- **Loss más bajo** (< 1.5)
- **SQL sintácticamente válido**
- **Estructura correcta** (SELECT, FROM, WHERE)
- **Menos repeticiones y errores**
- **Mejor comprensión de esquemas**

## 10. Prueba del Modelo Entrenado (Opcional)

Probamos el modelo con algunos ejemplos para verificar su funcionamiento

## 11. Carga del Modelo LoRA Entrenado

Cómo cargar y usar el modelo LoRA entrenado (para evaluación posterior)

In [ ]:
def cargar_modelo_lora_entrenado(model_path="./outputs/flan-t5-large-sql"):
    """
    Carga el modelo LoRA entrenado para evaluación
    """
    print("🔄 CARGANDO MODELO LORA ENTRENADO")
    print("=" * 40)
    
    try:
        # Verificar que existen los archivos
        import os
        if not os.path.exists(model_path):
            print(f"❌ No se encontró el directorio: {model_path}")
            return None, None
        
        # Cargar información del modelo
        info_path = f"{model_path}/model_info.json"
        if os.path.exists(info_path):
            with open(info_path, "r") as f:
                model_info = json.load(f)
            print(f"📋 Información del modelo encontrada:")
            print(f"   Modelo base: {model_info['base_model']}")
            print(f"   Loss final: {model_info.get('final_loss', 'N/A')}")
            print(f"   Épocas: {model_info['training_config']['num_epochs']}")
        
        # Cargar modelo base
        print("🧠 Cargando modelo base...")
        base_model = T5ForConditionalGeneration.from_pretrained(
            CONFIG["model_name"],
            torch_dtype=torch.float16,
            device_map="auto",
            load_in_8bit=True
        )
        
        # Cargar adaptadores LoRA
        print("🔧 Cargando adaptadores LoRA...")
        from peft import PeftModel
        lora_model = PeftModel.from_pretrained(base_model, model_path)
        
        # Cargar tokenizer
        print("🔤 Cargando tokenizer...")
        trained_tokenizer = T5Tokenizer.from_pretrained(model_path)
        
        print("✅ Modelo LoRA cargado exitosamente")
        print("🎯 Listo para inferencia y evaluación")
        
        return lora_model, trained_tokenizer
        
    except Exception as e:
        print(f"❌ Error cargando modelo LoRA: {e}")
        print("💡 Asegúrate de que el entrenamiento se completó correctamente")
        return None, None

# Ejemplo de carga (descomenta para usar)
# trained_model, trained_tokenizer = cargar_modelo_lora_entrenado()

print("📝 Para cargar el modelo entrenado:")
print("   1. Descomenta la línea de carga arriba")
print("   2. Asegúrate de que existe el directorio de salida")
print("   3. Usa trained_model y trained_tokenizer para inferencia")

print("\n🔄 GUÍA DE RECUPERACIÓN SI NO SE GUARDÓ:")
print("1. Ejecuta solo las celdas de preparación (1-8)")
print("2. Salta el entrenamiento y carga un modelo pre-existente")
print("3. O re-entrena con configuración optimizada de guardado")

In [ ]:
def probar_modelo_mejorado():
    """Prueba el modelo MEJORADO con ejemplos más desafiantes"""
    print("🧪 PROBANDO MODELO FLAN-T5-LARGE MEJORADO")
    print("=" * 60)
    
    # Ejemplos de prueba más complejos y realistas
    test_cases = [
        {
            "sql_context": "CREATE TABLE users (id INT, name VARCHAR(50), age INT, city VARCHAR(50), email VARCHAR(100));",
            "sql_prompt": "Get all users older than 25 from New York"
        },
        {
            "sql_context": "CREATE TABLE products (id INT, name VARCHAR(100), price DECIMAL, category VARCHAR(50)); CREATE TABLE orders (id INT, product_id INT, quantity INT, order_date DATE);",
            "sql_prompt": "Find the total revenue for each product category"
        },
        {
            "sql_context": "CREATE TABLE employees (id INT, name VARCHAR(50), department VARCHAR(50), salary DECIMAL, hire_date DATE);",
            "sql_prompt": "What is the average salary by department for employees hired after 2020?"
        },
        {
            "sql_context": "CREATE TABLE customers (id INT, name VARCHAR(50), country VARCHAR(50)); CREATE TABLE orders (id INT, customer_id INT, total DECIMAL, order_date DATE);",
            "sql_prompt": "Show the top 5 customers by total order value"
        },
        {
            "sql_context": "CREATE TABLE students (id INT, name VARCHAR(50), grade DECIMAL); CREATE TABLE courses (id INT, name VARCHAR(50), credits INT); CREATE TABLE enrollments (student_id INT, course_id INT, semester VARCHAR(20));",
            "sql_prompt": "List all students enrolled in courses with more than 3 credits"
        }
    ]
    
    print(f"🎯 Probando con {len(test_cases)} casos de prueba MEJORADOS")
    print(f"📝 Usando formato de prompt mejorado...")
    
    resultados_exitosos = 0
    
    for i, test in enumerate(test_cases, 1):
        print(f"\n🧪 PRUEBA {i}:")
        print(f"📋 Schema: {test['sql_context'][:80]}...")
        print(f"❓ Pregunta: {test['sql_prompt']}")
        
        try:
            # Formatear input usando la función MEJORADA
            input_text, _ = formatear_para_t5_mejorado(
                test['sql_context'], 
                test['sql_prompt'], 
                ""  # SQL vacío para la prueba
            )
            
            # Tokenizar
            inputs = tokenizer(
                input_text,
                return_tensors="pt",
                truncation=True,
                max_length=CONFIG["max_input_length"]
            )
            
            # Mover al device del modelo
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            
            # Generar SQL con parámetros mejorados
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=CONFIG["max_target_length"],
                    num_beams=5,                    # Más beams para mejor calidad
                    early_stopping=True,
                    do_sample=False,
                    temperature=0.7,               # Temperatura para variabilidad controlada
                    repetition_penalty=1.1,       # Evitar repeticiones
                    length_penalty=1.0,           # Penalización por longitud
                    no_repeat_ngram_size=3,       # Evitar n-gramas repetidos
                    pad_token_id=tokenizer.pad_token_id
                )
            
            # Decodificar resultado
            generated_sql = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Limpiar el resultado (quitar el prompt si aparece)
            if "SQL query:" in generated_sql:
                generated_sql = generated_sql.split("SQL query:")[-1].strip()
            
            # Validación básica de SQL
            sql_valido = False
            sql_keywords = ['SELECT', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'WITH']
            if any(keyword in generated_sql.upper() for keyword in sql_keywords):
                sql_valido = True
                resultados_exitosos += 1
            
            # Mostrar resultado con evaluación
            status = "✅ VÁLIDO" if sql_valido else "❌ INVÁLIDO"
            print(f"{status} SQL generado: {generated_sql}")
            
            # Análisis adicional
            if sql_valido:
                if len(generated_sql) > 20 and ';' in generated_sql:
                    print("   🎯 Estructura SQL correcta detectada")
                else:
                    print("   ⚠️ SQL válido pero posiblemente incompleto")
            else:
                print("   ❌ No se detectó estructura SQL válida")
            
        except Exception as e:
            print(f"❌ Error en generación: {e}")
        
        print("-" * 60)
    
    # Resumen de resultados
    tasa_exito = (resultados_exitosos / len(test_cases)) * 100
    print(f"\n📊 RESUMEN DE RESULTADOS:")
    print(f"✅ Predicciones válidas: {resultados_exitosos}/{len(test_cases)} ({tasa_exito:.1f}%)")
    
    if tasa_exito >= 80:
        print("🎉 ¡EXCELENTE! El modelo tiene muy buena calidad")
    elif tasa_exito >= 60:
        print("👍 BUENO: El modelo funciona bien, algunas mejoras posibles")
    elif tasa_exito >= 40:
        print("⚠️ REGULAR: El modelo necesita más entrenamiento")
    else:
        print("❌ POBRE: El modelo necesita revisión de configuración")
    
    return tasa_exito

# Función de comparación entre modelos
def comparar_modelos():
    """Compara el rendimiento entre diferentes versiones del modelo"""
    print("🔄 COMPARACIÓN DE MODELOS")
    print("=" * 40)
    
    # Caso de prueba simple para comparación
    test_simple = {
        "sql_context": "CREATE TABLE users (id INT, name VARCHAR(50), age INT);",
        "sql_prompt": "Get all users older than 30"
    }
    
    print(f"📝 Caso de prueba:")
    print(f"Schema: {test_simple['sql_context']}")
    print(f"Pregunta: {test_simple['sql_prompt']}")
    print(f"SQL esperado: SELECT * FROM users WHERE age > 30;")
    
    try:
        # Probar con formato mejorado
        input_text, _ = formatear_para_t5_mejorado(
            test_simple['sql_context'], 
            test_simple['sql_prompt'], 
            ""
        )
        
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=3,
                early_stopping=True,
                do_sample=False
            )
        
        generated_sql = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "SQL query:" in generated_sql:
            generated_sql = generated_sql.split("SQL query:")[-1].strip()
            
        print(f"\n🧪 Resultado del modelo MEJORADO:")
        print(f"SQL generado: {generated_sql}")
        
        # Análisis de calidad
        if "SELECT" in generated_sql.upper() and "WHERE" in generated_sql.upper() and "age" in generated_sql.lower():
            print("✅ MEJORA DETECTADA: Estructura SQL correcta con condición WHERE")
        else:
            print("⚠️ Aún necesita mejoras en estructura SQL")
            
    except Exception as e:
        print(f"❌ Error en comparación: {e}")

# Ejecutar pruebas MEJORADAS (descomenta para usar)
print("📝 Para probar el modelo MEJORADO:")
print("   1. Descomenta las líneas de prueba abajo")
print("   2. Ejecuta probar_modelo_mejorado()")
print("   3. Compara con comparar_modelos()")

# probar_modelo_mejorado()
# comparar_modelos()

print("\n🔄 INSTRUCCIONES DE USO:")
print("✅ El modelo mejorado debería generar SQL mucho más preciso")
print("🎯 Esperamos ver mejoras en:")
print("   • Estructura SQL válida (SELECT, FROM, WHERE)")
print("   • Uso correcto de condiciones")
print("   • Sintaxis SQL apropiada")
print("   • Menos repeticiones y errores")